# 02 — Saudi Household Data Integration and Validation

## Purpose

This notebook builds the first authoritative Saudi household electricity panel directly from:

1. Original GASTAT 2019–2022 Excel tables.
2. The independently supplied 2017–2022 long household CSV.

It never reads old merged panels, ML features, predictions, or metrics.

## Scientific source-selection rule

- Every overlap is compared before selection.
- A difference greater than `0.001` in the reported unit is unresolved and stops integration.
- When the original GASTAT workbook and CSV agree within tolerance, the GASTAT value is selected and the agreeing CSV value is retained in the overlap audit.
- For 2017–2018, the CSV is selected because it is the only supplied authoritative source.
- National-total rows are used for reconciliation but are excluded from the thirteen-region panels.

## Outputs

Interim extracts, annual and seasonal panels, detailed overlap evidence, merge documentation, and validation results.


## 1. Imports, Paths, and Canonical Definitions


In [ ]:
from datetime import datetime
from pathlib import Path
import hashlib
import re

import numpy as np
import openpyxl
import pandas as pd

RUN_STARTED = datetime.now().astimezone()
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "notebooks":
    raise RuntimeError(
        "Run this notebook from Saudi-Energy-Forecasting/notebooks/. "
        f"Current directory: {NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
GASTAT_DIR = RAW_DIR / "old_thesis_data"
SAUDI_ENERGY_DIR = RAW_DIR / "saudi_energy"
REFERENCE_DIR = RAW_DIR / "technical_work_reference"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in (INTERIM_DIR, PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

HOUSEHOLD_CSV_NAME = (
    "houses-consumption-and-cost-of-electricity-in-the-administrative-regions- (2).csv"
)
HOUSEHOLD_CSV_PATH = SAUDI_ENERGY_DIR / HOUSEHOLD_CSV_NAME

GASTAT_TABLES = {
    2019: {
        "file": "HouseholdEnergyStatistics2019En.xlsx",
        "sheet": "76",
        "table": "Table 37",
        "data_rows": (8, 20),
    },
    2020: {
        "file": "HouseholdEnergyStatistics2020En.xlsx",
        "sheet": "25",
        "table": "Houses Consumption and Cost",
        "data_rows": (8, 20),
    },
    2021: {
        "file": "HouseholdEnergyStatistics2021En.xlsx",
        "sheet": "27",
        "table": "Houses Consumption and Cost",
        "data_rows": (8, 20),
    },
    2022: {
        "file": "HouseholdEnergyStatistics2022En.xlsx",
        "sheet": "1-2",
        "table": "Table 1-2",
        "data_rows": (8, 20),
    },
}

CANONICAL_REGIONS = [
    "Riyadh",
    "Makkah",
    "Madinah",
    "Al-Qassim",
    "Eastern Region",
    "Asir",
    "Tabuk",
    "Hail",
    "Northern Borders",
    "Jazan",
    "Najran",
    "Al-Bahah",
    "Al-Jouf",
]

REGION_MAP = {
    "Riyadh": "Riyadh",
    "Makkah": "Makkah",
    "Madinah": "Madinah",
    "Qassim": "Al-Qassim",
    "Al Qassim": "Al-Qassim",
    "Al-Qassim": "Al-Qassim",
    "Eastern Region": "Eastern Region",
    "Eastern": "Eastern Region",
    "Aseer": "Asir",
    "Asir": "Asir",
    "Tabuk": "Tabuk",
    "Hail": "Hail",
    "Northern Border": "Northern Borders",
    "Northern Borders": "Northern Borders",
    "Jazan": "Jazan",
    "Najran": "Najran",
    "Al-Baha": "Al-Bahah",
    "Al Bahah": "Al-Bahah",
    "Al-Bahah": "Al-Bahah",
    "Al Jouf": "Al-Jouf",
    "Al-Jawf": "Al-Jouf",
    "Al-Jouf": "Al-Jouf",
}


def normalize_region(value):
    """Return a canonical administrative-region label without guessing unknown names."""
    if pd.isna(value):
        return pd.NA
    cleaned = re.sub(r"\s+", " ", str(value).strip().replace("–", "-").replace("—", "-"))
    return REGION_MAP.get(cleaned, cleaned)


def read_csv_flexible(path):
    """Read a supplied CSV while detecting comma/semicolon delimiters."""
    for encoding in ("utf-8-sig", "utf-8", "cp1256", "latin-1"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Unable to decode {path}")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def extract_gastat_household(year):
    """Extract one original GASTAT regional household table into canonical long form."""
    setting = GASTAT_TABLES[year]
    path = GASTAT_DIR / setting["file"]
    if not path.is_file():
        raise FileNotFoundError(path)
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    if setting["sheet"] not in workbook.sheetnames:
        raise KeyError(f"Missing sheet {setting['sheet']} in {path.name}")
    sheet = workbook[setting["sheet"]]
    start_row, end_row = setting["data_rows"]
    records = []
    for excel_row, row in enumerate(
        sheet.iter_rows(min_row=start_row, max_row=end_row, values_only=True),
        start=start_row,
    ):
        values = list(row)
        if len(values) < 6 or values[1] is None:
            continue
        original_region = str(values[1]).strip()
        region = normalize_region(original_region)
        measures = [
            ("Winter", "Consumption", "kWh", values[2]),
            ("Winter", "Cost", "SAR", values[3]),
            ("Rest of the year", "Consumption", "kWh", values[4]),
            ("Rest of the year", "Cost", "SAR", values[5]),
        ]
        for season, measure, unit, raw_value in measures:
            records.append(
                {
                    "year": year,
                    "region_original": original_region,
                    "region": region,
                    "season": season,
                    "measure": measure,
                    "unit": unit,
                    "value": pd.to_numeric(raw_value, errors="coerce"),
                    "source_type": "GASTAT original workbook",
                    "source_file": setting["file"],
                    "source_sheet": setting["sheet"],
                    "source_table": setting["table"],
                    "source_excel_row": excel_row,
                }
            )
    result = pd.DataFrame(records)
    expected = len(CANONICAL_REGIONS) * 2 * 2
    if len(result) != expected:
        raise ValueError(f"{year} GASTAT extraction produced {len(result)} rows; expected {expected}")
    if set(result["region"]) != set(CANONICAL_REGIONS):
        raise ValueError(f"{year} GASTAT region coverage is not canonical")
    return result.sort_values(["year", "region", "season", "measure"]).reset_index(drop=True)


def read_household_csv():
    """Read the supplied 2017–2022 long household CSV independently."""
    if not HOUSEHOLD_CSV_PATH.is_file():
        raise FileNotFoundError(HOUSEHOLD_CSV_PATH)
    frame = read_csv_flexible(HOUSEHOLD_CSV_PATH)
    expected_columns = {
        "Year", "Administrative Region", "Season", "Measure and Unit", "Value"
    }
    if not expected_columns.issubset(frame.columns):
        raise ValueError(
            f"Household CSV columns changed. Found: {frame.columns.tolist()}"
        )
    frame = frame.rename(
        columns={
            "Year": "year",
            "Administrative Region": "region_original",
            "Season": "season",
            "Value": "value",
        }
    )
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["region"] = frame["region_original"].map(normalize_region)
    parsed = frame["Measure and Unit"].str.extract(
        r"(?P<measure>Consumption|Cost)\s*\((?P<unit>[^)]+)\)",
        expand=True,
    )
    frame["measure"] = parsed["measure"]
    frame["unit"] = parsed["unit"].replace({"KWh": "kWh", "kWh": "kWh"})
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    frame["source_type"] = "Independent supplied household CSV"
    frame["source_file"] = HOUSEHOLD_CSV_NAME
    frame["source_sheet"] = pd.NA
    frame["source_table"] = pd.NA
    frame["source_excel_row"] = pd.NA
    ordered = [
        "year", "region_original", "region", "season", "measure", "unit", "value",
        "source_type", "source_file", "source_sheet", "source_table", "source_excel_row",
    ]
    return frame[ordered].sort_values(
        ["year", "region", "season", "measure"]
    ).reset_index(drop=True)


print(f"Project root: {PROJECT_ROOT}")
print(f"Run started: {RUN_STARTED.isoformat(timespec='seconds')}")


## 2. Extract Original GASTAT Tables


In [ ]:
gastat_by_year = {}
for year in sorted(GASTAT_TABLES):
    extract = extract_gastat_household(year)
    output = INTERIM_DIR / f"gastat_{year}.csv"
    extract.to_csv(output, index=False)
    gastat_by_year[year] = extract
    print(f"{year}: {len(extract)} rows saved to {output.relative_to(PROJECT_ROOT)}")

gastat_all = pd.concat(gastat_by_year.values(), ignore_index=True)
display(gastat_all.head())


## 3. Read and Standardize the Independent 2017–2022 Household CSV


In [ ]:
household_csv_all = read_household_csv()
household_csv_regions = household_csv_all[
    household_csv_all["region_original"].astype(str).str.lower().ne("total")
].copy()
household_csv_totals = household_csv_all[
    household_csv_all["region_original"].astype(str).str.lower().eq("total")
].copy()

unknown = sorted(set(household_csv_regions["region"]) - set(CANONICAL_REGIONS))
if unknown:
    raise ValueError(
        "Unrecognized administrative-region labels require a documented decision: "
        f"{unknown}"
    )

csv_interim_path = INTERIM_DIR / "household_csv_2017_2022.csv"
household_csv_all.to_csv(csv_interim_path, index=False)

print(f"Rows including national totals: {len(household_csv_all)}")
print(f"Regional observations: {len(household_csv_regions)}")
print(f"Years: {sorted(household_csv_regions['year'].dropna().astype(int).unique())}")
display(
    household_csv_regions[["region_original", "region"]]
    .drop_duplicates()
    .sort_values("region")
)


## 4. Compare All Overlapping Observations


In [ ]:
keys = ["year", "region", "season", "measure", "unit"]
csv_overlap = household_csv_regions[
    household_csv_regions["year"].isin(GASTAT_TABLES)
].copy()

overlap_report = gastat_all[
    keys + [
        "value", "source_file", "source_sheet", "source_table", "source_excel_row"
    ]
].merge(
    csv_overlap[keys + ["value", "source_file"]],
    on=keys,
    how="outer",
    suffixes=("_gastat", "_csv"),
    indicator=True,
    validate="one_to_one",
)

overlap_report["absolute_difference"] = (
    overlap_report["value_gastat"] - overlap_report["value_csv"]
).abs()
overlap_report["percentage_difference"] = np.where(
    overlap_report["value_gastat"].abs() > 0,
    overlap_report["absolute_difference"] / overlap_report["value_gastat"].abs() * 100,
    np.nan,
)
ABSOLUTE_TOLERANCE = 1e-3
overlap_report["agreement_status"] = np.select(
    [
        overlap_report["_merge"].ne("both"),
        overlap_report["absolute_difference"].gt(ABSOLUTE_TOLERANCE),
    ],
    ["Missing from one source", "Conflict above tolerance"],
    default="Agreement within tolerance",
)
overlap_report["selected_source"] = np.where(
    overlap_report["agreement_status"].eq("Agreement within tolerance"),
    "GASTAT original workbook",
    "UNRESOLVED",
)
overlap_report["selected_value"] = np.where(
    overlap_report["selected_source"].eq("GASTAT original workbook"),
    overlap_report["value_gastat"],
    np.nan,
)
overlap_report["justification"] = np.where(
    overlap_report["selected_source"].eq("GASTAT original workbook"),
    "Original GASTAT publication selected; independent supplied CSV agrees within tolerance.",
    "No value selected: source discrepancy requires an explicit scientific decision.",
)

unresolved = overlap_report[
    overlap_report["selected_source"].eq("UNRESOLVED")
]
overlap_path = RESULTS_DIR / "overlap_report.csv"
overlap_report.to_csv(overlap_path, index=False)

print(overlap_report["agreement_status"].value_counts())
print(f"Maximum absolute difference: {overlap_report['absolute_difference'].max():.9f}")
if not unresolved.empty:
    display(unresolved)
    raise RuntimeError(
        f"{len(unresolved)} overlap observations are unresolved. "
        "Integration stopped without choosing a source."
    )


## 5. Select Values Transparently and Preserve Lineage


In [ ]:
# GASTAT is selected for 2019–2022 only after the overlap audit has passed.
selected_gastat = gastat_all.copy()
selected_gastat["selected_source"] = "GASTAT original workbook"
selected_gastat["selection_justification"] = (
    "Original GASTAT publication selected; independent supplied CSV agrees within tolerance."
)
selected_gastat["corroborating_source"] = HOUSEHOLD_CSV_NAME

# The CSV is the sole supplied authoritative source for 2017–2018.
selected_csv = household_csv_regions[
    household_csv_regions["year"].isin([2017, 2018])
].copy()
selected_csv["selected_source"] = "Independent supplied household CSV"
selected_csv["selection_justification"] = (
    "CSV selected because it is the only supplied authoritative source for 2017–2018; "
    "publisher/provenance metadata should be completed in the source registry."
)
selected_csv["corroborating_source"] = pd.NA

selected_long = pd.concat([selected_csv, selected_gastat], ignore_index=True, sort=False)
selected_long = selected_long.sort_values(
    ["year", "region", "season", "measure"]
).reset_index(drop=True)

expected_rows = 6 * 13 * 2 * 2
if len(selected_long) != expected_rows:
    raise ValueError(
        f"Selected long dataset contains {len(selected_long)} rows; expected {expected_rows}"
    )
duplicate_keys = selected_long.duplicated(keys).sum()
if duplicate_keys:
    raise ValueError(f"Selected long dataset contains {duplicate_keys} duplicate keys")
if selected_long["value"].isna().any():
    raise ValueError("Selected household values contain missing observations")
if (selected_long["value"] < 0).any():
    raise ValueError("Selected household values contain negative observations")

display(selected_long.head())


## 6. Build the Seasonal Panel


In [ ]:
seasonal_values = selected_long.pivot(
    index=["year", "region", "season"],
    columns="measure",
    values="value",
).reset_index()
seasonal_values = seasonal_values.rename(
    columns={"Consumption": "consumption_kwh", "Cost": "cost_sar"}
)

lineage = (
    selected_long.groupby(["year", "region", "season"], as_index=False)
    .agg(
        selected_source=("selected_source", lambda values: " | ".join(sorted(set(values)))),
        source_file=("source_file", lambda values: " | ".join(sorted(set(values.astype(str))))),
        source_sheet=("source_sheet", lambda values: " | ".join(sorted(set(values.dropna().astype(str))))),
        selection_justification=(
            "selection_justification",
            lambda values: " | ".join(sorted(set(values))),
        ),
        corroborating_source=(
            "corroborating_source",
            lambda values: " | ".join(sorted(set(values.dropna().astype(str)))),
        ),
    )
)
seasonal_panel = seasonal_values.merge(
    lineage,
    on=["year", "region", "season"],
    how="left",
    validate="one_to_one",
)
seasonal_panel["geographic_resolution"] = "Administrative region"
seasonal_panel["temporal_resolution"] = "Annual with two seasonal components"
seasonal_panel = seasonal_panel.sort_values(["year", "region", "season"]).reset_index(drop=True)

seasonal_path = PROCESSED_DIR / "saudi_household_admin_region_seasonal_panel.csv"
seasonal_panel.to_csv(seasonal_path, index=False)
print(f"Seasonal panel: {seasonal_panel.shape}")
display(seasonal_panel.head())


## 7. Build the Annual Panel Without Missing-as-Zero Logic


In [ ]:
annual_values = (
    seasonal_panel.groupby(["year", "region"], as_index=False)
    .agg(
        winter_consumption_kwh=(
            "consumption_kwh",
            lambda values: values.loc[
                seasonal_panel.loc[values.index, "season"].eq("Winter")
            ].iloc[0],
        ),
        rest_of_year_consumption_kwh=(
            "consumption_kwh",
            lambda values: values.loc[
                seasonal_panel.loc[values.index, "season"].eq("Rest of the year")
            ].iloc[0],
        ),
        winter_cost_sar=(
            "cost_sar",
            lambda values: values.loc[
                seasonal_panel.loc[values.index, "season"].eq("Winter")
            ].iloc[0],
        ),
        rest_of_year_cost_sar=(
            "cost_sar",
            lambda values: values.loc[
                seasonal_panel.loc[values.index, "season"].eq("Rest of the year")
            ].iloc[0],
        ),
        selected_source=("selected_source", "first"),
        source_file=("source_file", "first"),
        source_sheet=("source_sheet", "first"),
        selection_justification=("selection_justification", "first"),
        corroborating_source=("corroborating_source", "first"),
    )
)

component_columns = [
    "winter_consumption_kwh", "rest_of_year_consumption_kwh",
    "winter_cost_sar", "rest_of_year_cost_sar",
]
if annual_values[component_columns].isna().any().any():
    raise ValueError(
        "Annual values were not calculated because one or more seasonal components are missing"
    )

annual_values["annual_consumption_kwh"] = (
    annual_values["winter_consumption_kwh"]
    + annual_values["rest_of_year_consumption_kwh"]
)
annual_values["annual_cost_sar"] = (
    annual_values["winter_cost_sar"]
    + annual_values["rest_of_year_cost_sar"]
)
annual_values["average_cost_sar_per_kwh"] = (
    annual_values["annual_cost_sar"] / annual_values["annual_consumption_kwh"]
)
annual_values["winter_consumption_share_pct"] = (
    annual_values["winter_consumption_kwh"]
    / annual_values["annual_consumption_kwh"]
    * 100
)
annual_values["geographic_resolution"] = "Administrative region"
annual_values["temporal_resolution"] = "Annual"

annual_panel = annual_values.sort_values(["year", "region"]).reset_index(drop=True)
annual_path = PROCESSED_DIR / "saudi_household_admin_region_annual_panel.csv"
annual_panel.to_csv(annual_path, index=False)

print(f"Annual panel: {annual_panel.shape}")
display(annual_panel.head())


## 8. Validate Regional Coverage and National Totals


In [ ]:
validation_rows = []

coverage = annual_panel.groupby("year")["region"].nunique()
for year, count in coverage.items():
    validation_rows.append(
        {
            "check": f"{year} administrative-region coverage",
            "status": "PASS" if count == 13 else "FAIL",
            "observed": int(count),
            "expected": 13,
        }
    )

csv_total_pivot = household_csv_totals.pivot(
    index=["year", "season"],
    columns="measure",
    values="value",
).reset_index()
csv_total_pivot = csv_total_pivot.rename(
    columns={"Consumption": "reported_consumption_kwh", "Cost": "reported_cost_sar"}
)
regional_totals = (
    seasonal_panel.groupby(["year", "season"], as_index=False)
    .agg(
        calculated_consumption_kwh=("consumption_kwh", "sum"),
        calculated_cost_sar=("cost_sar", "sum"),
    )
)
national_reconciliation = regional_totals.merge(
    csv_total_pivot,
    on=["year", "season"],
    how="outer",
    validate="one_to_one",
)
national_reconciliation["consumption_difference"] = (
    national_reconciliation["calculated_consumption_kwh"]
    - national_reconciliation["reported_consumption_kwh"]
)
national_reconciliation["cost_difference"] = (
    national_reconciliation["calculated_cost_sar"]
    - national_reconciliation["reported_cost_sar"]
)
TOTAL_RECONCILIATION_TOLERANCE = 5.0
# Published totals are reported at whole kWh/SAR precision while some regional
# components contain decimals. Differences of at most five base units are
# treated as arithmetic rounding, never used to modify a regional observation.
national_reconciliation["status"] = np.where(
    national_reconciliation[["consumption_difference", "cost_difference"]]
    .abs()
    .max(axis=1)
    .le(TOTAL_RECONCILIATION_TOLERANCE),
    "PASS",
    "FAIL",
)

for row in national_reconciliation.itertuples(index=False):
    validation_rows.append(
        {
            "check": f"{row.year} {row.season} regional sum versus reported total",
            "status": row.status,
            "observed": max(abs(row.consumption_difference), abs(row.cost_difference)),
            "expected": (
                f"absolute difference <= {TOTAL_RECONCILIATION_TOLERANCE} "
                "kWh/SAR (published-total rounding tolerance)"
            ),
        }
    )

validation_table = pd.DataFrame(validation_rows)
display(validation_table)
if validation_table["status"].ne("PASS").any():
    raise RuntimeError(
        "Validation failed. Panels were written for inspection but must not be treated as validated."
    )


## 9. Write Merge and Validation Documentation


In [ ]:
merge_lines = [
    "# Household Source Merge Report",
    "",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    "",
    "## Authoritative inputs",
    "",
    "- Original GASTAT household workbooks for 2019–2022.",
    f"- `{HOUSEHOLD_CSV_NAME}` for 2017–2022.",
    "- No historical merged panel, ML feature file, prediction, model, or metric was used.",
    "",
    "## Merge key",
    "",
    "`year + canonical administrative region + season + measure + unit`",
    "",
    "## Overlap result",
    "",
    f"- Compared observations: {len(overlap_report):,}",
    f"- Unresolved observations: {len(unresolved):,}",
    f"- Maximum absolute difference: {overlap_report['absolute_difference'].max():.9f}",
    f"- Absolute tolerance: {ABSOLUTE_TOLERANCE}",
    "",
    "## Source selection",
    "",
    "- 2019–2022: original GASTAT workbook selected after the supplied CSV agreed within tolerance.",
    "- 2017–2018: supplied household CSV selected because it is the only authoritative source provided.",
    "- National total rows were used for reconciliation and excluded from regional panel rows.",
    "",
    "## Important limitation",
    "",
    "The publisher/provenance metadata for the supplied 2017–2018 CSV observations should be completed "
    "before publication. Their values are retained with explicit source labels rather than represented "
    "as GASTAT observations.",
]
merge_path = RESULTS_DIR / "merge_report.md"
merge_path.write_text("\n".join(merge_lines) + "\n", encoding="utf-8")

required_numeric_missing = int(
    annual_panel[
        [
            "winter_consumption_kwh",
            "rest_of_year_consumption_kwh",
            "winter_cost_sar",
            "rest_of_year_cost_sar",
            "annual_consumption_kwh",
            "annual_cost_sar",
        ]
    ].isna().sum().sum()
)
validation_lines = [
    "# Phase 1 Household Panel Validation Summary",
    "",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    "",
    f"- Annual panel rows: {len(annual_panel):,}",
    f"- Seasonal panel rows: {len(seasonal_panel):,}",
    f"- Years: {int(annual_panel['year'].min())}–{int(annual_panel['year'].max())}",
    f"- Administrative regions: {annual_panel['region'].nunique()}",
    f"- Missing required annual numeric values: {required_numeric_missing}",
    f"- Duplicate annual keys: {int(annual_panel.duplicated(['year','region']).sum())}",
    f"- Duplicate seasonal keys: {int(seasonal_panel.duplicated(['year','region','season']).sum())}",
    "",
    "## Check results",
    "",
]
for row in validation_table.itertuples(index=False):
    validation_lines.append(
        f"- **{row.status}** — {row.check}: observed `{row.observed}`, expected `{row.expected}`"
    )
validation_lines.extend(
    [
        "",
        "All annual values were calculated only when both seasonal components were present. "
        "No missing component was replaced with zero.",
        "",
        "Reported national totals differ from summed regional components by at most three "
        "kWh/SAR units in the supplied sources. These immaterial arithmetic differences are "
        "classified as published-total rounding; regional observations were not modified.",
    ]
)
validation_path = RESULTS_DIR / "validation_summary.md"
validation_path.write_text("\n".join(validation_lines) + "\n", encoding="utf-8")

required = [
    *(INTERIM_DIR / f"gastat_{year}.csv" for year in GASTAT_TABLES),
    csv_interim_path,
    annual_path,
    seasonal_path,
    merge_path,
    overlap_path,
    validation_path,
]
for path in required:
    if not path.is_file() or path.stat().st_size == 0:
        raise IOError(f"Required output missing or empty: {path}")

if len(annual_panel) != 78 or len(seasonal_panel) != 156:
    raise RuntimeError("Final panel dimensions are not the expected 78 annual and 156 seasonal rows")

print("FINAL PHASE 1 INTEGRATION VALIDATION PASSED")
for path in required:
    print("-", path.relative_to(PROJECT_ROOT))


## Stop point

The validated panels contain no forecasting features and no trained models. Feature engineering and forecasting are intentionally deferred until the Phase 1 outputs, overlap evidence, source policy, and documented limitation for the 2017–2018 provenance have been reviewed and approved.
